# WP1 — WSI & Report Exploration

Goals:
1. Inventory the `.svs` files in `TUMUntera/` (count, sizes, per-case grouping).
2. Load the label spreadsheet (`slide_ids` / `english_reports`, or legacy `slide_id` / `english_report`).
3. Check the `case_class` (A–E) distribution and report richness per class.
4. Map report sections (macro / micro / IHC / molecular / diagnosis) to graph nodes.

Run inside `dominik_mlmi` on a compute node. Paths come from `configs/paths.yaml`.

In [ ]:
from pathlib import Path
import yaml
import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
cfg = yaml.safe_load((REPO / 'configs' / 'paths.yaml').read_text())
data_dir = Path(cfg['cluster']['data_dir'])
labels_path = Path(cfg['cluster']['labels_xlsx'])
print('data:', data_dir)
print('labels:', labels_path)

In [ ]:
svs = sorted(data_dir.rglob('*.svs')) if data_dir.exists() else []
print(f'{len(svs)} .svs files')
for p in svs[:5]:
    print(' ', p.name, f'{p.stat().st_size / 1e9:.2f} GB')

In [ ]:
if labels_path.exists():
    df = pd.read_excel(labels_path)
    print('rows:', len(df), '| cols:', list(df.columns))
    display(df.head())
    if 'case_class' in df.columns:
        display(df['case_class'].value_counts())

In [ ]:
# Report richness per class (supports english_reports or english_report column).
report_col = next(
    (c for c in ('english_reports', 'english_report') if c in df.columns),
    None,
)
if labels_path.exists() and report_col and 'case_class' in df.columns:
    df['report_len'] = df[report_col].astype(str).str.len()
    display(df.groupby('case_class')['report_len'].agg(['count', 'mean', 'min', 'max']))

In [ ]:
# Open one slide with openslide to confirm magnification / level dims (WP2 prep).
try:
    import openslide
    if svs:
        slide = openslide.OpenSlide(str(svs[0]))
        print('dimensions:', slide.dimensions)
        print('level_count:', slide.level_count)
        print('level_dims:', slide.level_dimensions)
        print('mpp_x:', slide.properties.get('openslide.mpp-x'))
except ImportError:
    print('openslide not installed in this env')